In [ ]:
# 多滞后期XGBoost风力发电预测模型

本notebook实现了滞后1到滞后16的XGBoost模型训练和预测，使用90%数据作为训练集，10%作为测试集。
每个滞后期模型的预测结果分别保存到不同的CSV文件中，最后合并成一个综合的CSV文件。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 设置matplotlib中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
## 1. 多滞后期预测器类定义


In [ ]:
class MultiLagWindPowerPredictor:
    def __init__(self, lag_steps=16):
        """
        初始化多滞后期预测器
        
        Parameters:
        lag_steps: int, 最大滞后期数（1到lag_steps）
        """
        self.lag_steps = lag_steps
        self.models = {}  # 存储每个滞后期的模型
        self.scalers = {}  # 存储每个滞后期的标准化器
        self.feature_names = {}  # 存储每个滞后期的特征名称
        
    def create_lag_features(self, data, lag):
        """
        为指定滞后期创建特征
        
        Parameters:
        data: pd.DataFrame, 原始数据
        lag: int, 滞后期
        
        Returns:
        pd.DataFrame, 处理后的特征数据
        """
        features = data.copy()
        
        # 创建滞后功率特征
        if 'wp_true' in features.columns:
            features[f'power_lag_{lag}'] = features['wp_true'].shift(lag)
            
        # 创建风速特征的滞后和差分
        heights = [10, 100, 200]
        for height in heights:
            # 风速列
            ws_cols = [f'ws{height}_{i}' for i in range(1, 16) if f'ws{height}_{i}' in features.columns]
            
            # 风速差分特征
            for i in range(len(ws_cols)-1):
                features[f'ws{height}_diff_{i}'] = features[ws_cols[i+1]] - features[ws_cols[i]]
                
            # 风速时间差分特征
            for col in ws_cols:
                features[f'{col}_diff_prev1'] = features[col] - features[col].shift(1)
                features[f'{col}_diff_prev2'] = features[col] - features[col].shift(2)
                features[f'{col}_diff_prev3'] = features[col] - features[col].shift(3)
                
        # 创建目标变量（功率差）
        if 'wp_true' in features.columns:
            features[f'power_diff_{lag}'] = features['wp_true'] - features['wp_true'].shift(lag)
            
        return features
    
    def prepare_data_for_lag(self, data, lag):
        """
        为指定滞后期准备训练数据
        
        Parameters:
        data: pd.DataFrame, 原始数据
        lag: int, 滞后期
        
        Returns:
        X: pd.DataFrame, 特征数据
        y: pd.Series, 目标变量
        """
        # 创建滞后特征
        features = self.create_lag_features(data, lag)
        
        # 删除包含NaN的行
        features_clean = features.dropna()
        
        # 选择数值特征（排除目标变量和原始功率）
        exclude_cols = ['wp_true', f'power_diff_{lag}']
        numeric_cols = features_clean.select_dtypes(include=[np.number]).columns
        feature_cols = [col for col in numeric_cols if col not in exclude_cols]
        
        X = features_clean[feature_cols]
        y = features_clean[f'power_diff_{lag}']
        
        return X, y


In [ ]:
# 继续添加类的训练方法
def train_all_models(self, data, test_size=0.1, random_state=42):
    """
    训练所有滞后期的模型
    
    Parameters:
    data: pd.DataFrame, 原始数据
    test_size: float, 测试集比例
    random_state: int, 随机种子
    
    Returns:
    dict, 训练结果统计
    """
    train_results = {}
    
    print(f"开始训练滞后1到滞后{self.lag_steps}的模型...")
    
    for lag in range(1, self.lag_steps + 1):
        print(f"\\n--- 训练滞后{lag}模型 ---")
        
        try:
            # 准备数据
            X, y = self.prepare_data_for_lag(data, lag)
            
            if X.empty or y.empty:
                print(f"警告：滞后{lag}模型的数据为空，跳过训练")
                continue
            
            print(f"滞后{lag}模型：数据形状 X: {X.shape}, y: {y.shape}")
            
            # 分割训练集和测试集
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=random_state, shuffle=False
            )
            
            # 标准化特征
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # 训练XGBoost模型
            model = xgb.XGBRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                min_child_weight=1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=random_state,
                n_jobs=-1
            )
            
            model.fit(X_train_scaled, y_train)
            
            # 预测
            y_pred = model.predict(X_test_scaled)
            
            # 计算评估指标
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            
            # 保存模型和相关信息
            self.models[lag] = model
            self.scalers[lag] = scaler
            self.feature_names[lag] = list(X.columns)
            
            # 记录训练结果
            train_results[lag] = {
                'mse': mse,
                'mae': mae,
                'r2': r2,
                'train_size': len(X_train),
                'test_size': len(X_test),
                'features_count': len(X.columns)
            }
            
            print(f"滞后{lag}模型训练完成：")
            print(f"  MSE: {mse:.4f}")
            print(f"  MAE: {mae:.4f}")
            print(f"  R²: {r2:.4f}")
            print(f"  训练集大小: {len(X_train)}, 测试集大小: {len(X_test)}")
            
        except Exception as e:
            print(f"滞后{lag}模型训练失败: {e}")
            continue
    
    print(f"\\n所有模型训练完成！成功训练了{len(self.models)}个模型")
    return train_results

# 将方法添加到类中
MultiLagWindPowerPredictor.train_all_models = train_all_models


In [ ]:
# 继续添加类的预测和保存方法
def predict_all_models(self, data, output_dir='predictions'):
    """
    使用所有训练好的模型进行预测
    
    Parameters:
    data: pd.DataFrame, 预测数据
    output_dir: str, 输出目录
    
    Returns:
    dict, 所有模型的预测结果
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    all_predictions = {}
    
    print(f"开始使用{len(self.models)}个模型进行预测...")
    
    for lag in sorted(self.models.keys()):
        print(f"\\n--- 滞后{lag}模型预测 ---")
        
        try:
            # 准备数据
            features = self.create_lag_features(data, lag)
            
            # 选择特征
            X = features[self.feature_names[lag]]
            
            # 删除包含NaN的行
            valid_indices = X.dropna().index
            X_valid = X.loc[valid_indices]
            
            if X_valid.empty:
                print(f"警告：滞后{lag}模型的预测数据为空")
                continue
            
            # 标准化
            X_scaled = self.scalers[lag].transform(X_valid)
            
            # 预测
            predictions = self.models[lag].predict(X_scaled)
            
            # 重构实际功率预测值
            power_lag_col = f'power_lag_{lag}'
            if power_lag_col in features.columns:
                power_predictions = features.loc[valid_indices, power_lag_col] + predictions
            else:
                power_predictions = predictions
            
            # 创建预测结果DataFrame
            result_df = pd.DataFrame({
                'index': valid_indices,
                'power_diff_pred': predictions,
                'power_pred': power_predictions,
                'lag': lag
            })
            
            # 如果有真实值，添加到结果中
            if 'wp_true' in data.columns:
                result_df['wp_true'] = data.loc[valid_indices, 'wp_true']
            
            # 保存到CSV
            csv_filename = os.path.join(output_dir, f'predictions_lag_{lag}.csv')
            result_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
            
            all_predictions[lag] = result_df
            
            print(f"滞后{lag}模型预测完成，结果保存到: {csv_filename}")
            print(f"  预测样本数: {len(result_df)}")
            
        except Exception as e:
            print(f"滞后{lag}模型预测失败: {e}")
            continue
    
    return all_predictions

# 将预测方法添加到类中
MultiLagWindPowerPredictor.predict_all_models = predict_all_models


In [ ]:
# 继续添加合并预测和模型保存方法
def merge_predictions(self, all_predictions, output_dir='predictions'):
    """
    合并所有模型的预测结果
    
    Parameters:
    all_predictions: dict, 所有模型的预测结果
    output_dir: str, 输出目录
    
    Returns:
    pd.DataFrame, 合并后的预测结果
    """
    if not all_predictions:
        print("没有可合并的预测结果")
        return pd.DataFrame()
    
    # 合并所有预测结果
    merged_df = pd.concat(all_predictions.values(), ignore_index=True)
    
    # 按索引和滞后期排序
    merged_df = merged_df.sort_values(['index', 'lag'])
    
    # 保存合并结果
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    merged_filename = os.path.join(output_dir, f'merged_predictions_{timestamp}.csv')
    merged_df.to_csv(merged_filename, index=False, encoding='utf-8-sig')
    
    print(f"\\n所有预测结果已合并，保存到: {merged_filename}")
    print(f"合并结果包含 {len(merged_df)} 条记录")
    
    return merged_df

def save_models(self, save_dir='models'):
    """
    保存所有训练好的模型
    
    Parameters:
    save_dir: str, 保存目录
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    for lag in self.models.keys():
        model_path = os.path.join(save_dir, f'model_lag_{lag}.joblib')
        scaler_path = os.path.join(save_dir, f'scaler_lag_{lag}.joblib')
        features_path = os.path.join(save_dir, f'features_lag_{lag}.joblib')
        
        joblib.dump(self.models[lag], model_path)
        joblib.dump(self.scalers[lag], scaler_path)
        joblib.dump(self.feature_names[lag], features_path)
    
    print(f"所有模型已保存到目录: {save_dir}")

def load_models(self, load_dir='models'):
    """
    加载所有训练好的模型
    
    Parameters:
    load_dir: str, 加载目录
    """
    if not os.path.exists(load_dir):
        print(f"模型目录不存在: {load_dir}")
        return
    
    loaded_count = 0
    for lag in range(1, self.lag_steps + 1):
        model_path = os.path.join(load_dir, f'model_lag_{lag}.joblib')
        scaler_path = os.path.join(load_dir, f'scaler_lag_{lag}.joblib')
        features_path = os.path.join(load_dir, f'features_lag_{lag}.joblib')
        
        if all(os.path.exists(path) for path in [model_path, scaler_path, features_path]):
            self.models[lag] = joblib.load(model_path)
            self.scalers[lag] = joblib.load(scaler_path)
            self.feature_names[lag] = joblib.load(features_path)
            loaded_count += 1
    
    print(f"成功加载{loaded_count}个模型")

# 将方法添加到类中
MultiLagWindPowerPredictor.merge_predictions = merge_predictions
MultiLagWindPowerPredictor.save_models = save_models
MultiLagWindPowerPredictor.load_models = load_models


In [ ]:
## 2. 数据加载和预处理


In [ ]:
# 加载数据（请根据实际情况修改文件路径）
# 这里假设你有一个CSV文件包含风力发电数据
# 请替换为你的实际数据文件路径

# 示例数据路径
data_file = 'your_wind_power_data.csv'  # 请替换为实际文件路径

# 如果需要从项目中的示例数据开始
# data_file = 'ecmwf_data_ws_beijing_15min_middle_interpolated_robust.csv'

try:
    # 读取数据
    data = pd.read_csv(data_file)
    print(f"数据加载成功，形状: {data.shape}")
    print(f"列名: {list(data.columns)}")
    
    # 显示数据基本信息
    print("\\n数据基本信息:")
    print(data.info())
    
    # 显示前几行
    print("\\n前5行数据:")
    print(data.head())
    
except FileNotFoundError:
    print(f"文件未找到: {data_file}")
    print("请确保数据文件路径正确，或者使用以下代码生成示例数据:")
    
    # 生成示例数据用于演示
    np.random.seed(42)
    n_samples = 1000
    
    # 创建示例数据
    sample_data = {
        'wp_true': np.random.normal(50, 20, n_samples),  # 风电功率
    }
    
    # 添加不同高度的风速数据
    for height in [10, 100, 200]:
        for i in range(1, 16):
            sample_data[f'ws{height}_{i}'] = np.random.normal(8, 3, n_samples)
    
    data = pd.DataFrame(sample_data)
    print(f"\\n生成了示例数据，形状: {data.shape}")
    print("注意：这是示例数据，请替换为你的实际数据")


In [ ]:
## 3. 模型训练


In [ ]:
# 创建多滞后期预测器
predictor = MultiLagWindPowerPredictor(lag_steps=16)

# 训练所有模型（滞后1到滞后16）
print("开始训练多滞后期模型...")
training_results = predictor.train_all_models(data, test_size=0.1, random_state=42)

# 显示训练结果汇总
if training_results:
    print("\\n=== 训练结果汇总 ===")
    results_df = pd.DataFrame(training_results).T
    results_df.index.name = 'lag'
    print(results_df)
    
    # 保存训练结果
    results_df.to_csv('training_results.csv', encoding='utf-8-sig')
    print("\\n训练结果已保存到 training_results.csv")
else:
    print("没有成功训练任何模型，请检查数据")


In [ ]:
## 4. 模型预测


In [ ]:
# 使用所有训练好的模型进行预测
print("开始预测...")
all_predictions = predictor.predict_all_models(data, output_dir='predictions')

# 合并所有预测结果
merged_predictions = predictor.merge_predictions(all_predictions, output_dir='predictions')

# 显示合并结果的基本信息
if not merged_predictions.empty:
    print("\\n=== 合并预测结果信息 ===")
    print(f"总记录数: {len(merged_predictions)}")
    print(f"滞后期范围: {merged_predictions['lag'].min()} 到 {merged_predictions['lag'].max()}")
    print(f"\\n各滞后期预测数量:")
    print(merged_predictions['lag'].value_counts().sort_index())
    
    # 显示预测结果的前几行
    print("\\n前10行预测结果:")
    print(merged_predictions.head(10))


In [ ]:
## 5. 结果可视化


In [ ]:
# 可视化训练结果
if training_results:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 准备数据
    lags = list(training_results.keys())
    mses = [training_results[lag]['mse'] for lag in lags]
    maes = [training_results[lag]['mae'] for lag in lags]
    r2s = [training_results[lag]['r2'] for lag in lags]
    
    # MSE
    axes[0, 0].plot(lags, mses, 'bo-')
    axes[0, 0].set_title('各滞后期模型的MSE')
    axes[0, 0].set_xlabel('滞后期')
    axes[0, 0].set_ylabel('MSE')
    axes[0, 0].grid(True)
    
    # MAE
    axes[0, 1].plot(lags, maes, 'ro-')
    axes[0, 1].set_title('各滞后期模型的MAE')
    axes[0, 1].set_xlabel('滞后期')
    axes[0, 1].set_ylabel('MAE')
    axes[0, 1].grid(True)
    
    # R²
    axes[1, 0].plot(lags, r2s, 'go-')
    axes[1, 0].set_title('各滞后期模型的R²')
    axes[1, 0].set_xlabel('滞后期')
    axes[1, 0].set_ylabel('R²')
    axes[1, 0].grid(True)
    
    # 训练样本数
    train_sizes = [training_results[lag]['train_size'] for lag in lags]
    axes[1, 1].bar(lags, train_sizes, alpha=0.7)
    axes[1, 1].set_title('各滞后期模型的训练样本数')
    axes[1, 1].set_xlabel('滞后期')
    axes[1, 1].set_ylabel('训练样本数')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_results_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("训练结果可视化已保存到 training_results_visualization.png")


In [ ]:
# 可视化预测结果（如果有真实值）
if not merged_predictions.empty and 'wp_true' in merged_predictions.columns:
    # 选择几个滞后期进行可视化
    sample_lags = [1, 4, 8, 12, 16]
    available_lags = [lag for lag in sample_lags if lag in merged_predictions['lag'].values]
    
    if available_lags:
        fig, axes = plt.subplots(len(available_lags), 1, figsize=(15, 4*len(available_lags)))
        
        if len(available_lags) == 1:
            axes = [axes]
        
        for i, lag in enumerate(available_lags):
            lag_data = merged_predictions[merged_predictions['lag'] == lag].head(100)  # 只显示前100个点
            
            axes[i].plot(lag_data['wp_true'], label='真实值', alpha=0.7)
            axes[i].plot(lag_data['power_pred'], label='预测值', alpha=0.7)
            axes[i].set_title(f'滞后{lag}模型：真实值 vs 预测值')
            axes[i].set_xlabel('样本索引')
            axes[i].set_ylabel('功率')
            axes[i].legend()
            axes[i].grid(True)
        
        plt.tight_layout()
        plt.savefig('prediction_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("预测结果对比可视化已保存到 prediction_comparison.png")


In [ ]:
## 6. 保存和加载模型


In [ ]:
# 保存所有训练好的模型
predictor.save_models(save_dir='models')

# 演示如何加载模型
print("\\n演示模型加载:")
new_predictor = MultiLagWindPowerPredictor(lag_steps=16)
new_predictor.load_models(load_dir='models')
print(f"新预测器加载了 {len(new_predictor.models)} 个模型")


In [ ]:
## 7. 性能分析和总结


In [ ]:
# 性能分析
if training_results:
    print("\\n=== 性能分析总结 ===")
    
    # 找出表现最好的模型
    best_mse_lag = min(training_results.keys(), key=lambda x: training_results[x]['mse'])
    best_mae_lag = min(training_results.keys(), key=lambda x: training_results[x]['mae'])
    best_r2_lag = max(training_results.keys(), key=lambda x: training_results[x]['r2'])
    
    print(f"最佳MSE模型: 滞后{best_mse_lag} (MSE: {training_results[best_mse_lag]['mse']:.4f})")
    print(f"最佳MAE模型: 滞后{best_mae_lag} (MAE: {training_results[best_mae_lag]['mae']:.4f})")
    print(f"最佳R²模型: 滞后{best_r2_lag} (R²: {training_results[best_r2_lag]['r2']:.4f})")
    
    # 平均性能
    avg_mse = np.mean([training_results[lag]['mse'] for lag in training_results.keys()])
    avg_mae = np.mean([training_results[lag]['mae'] for lag in training_results.keys()])
    avg_r2 = np.mean([training_results[lag]['r2'] for lag in training_results.keys()])
    
    print(f"\\n平均性能:")
    print(f"  平均MSE: {avg_mse:.4f}")
    print(f"  平均MAE: {avg_mae:.4f}")
    print(f"  平均R²: {avg_r2:.4f}")
    
    # 性能趋势分析
    print(f"\\n性能趋势:")
    lags = sorted(training_results.keys())
    if len(lags) > 1:
        mse_trend = "递增" if training_results[lags[-1]]['mse'] > training_results[lags[0]]['mse'] else "递减"
        print(f"  MSE从滞后{lags[0]}到滞后{lags[-1]}总体{mse_trend}")
        
        r2_trend = "递增" if training_results[lags[-1]]['r2'] > training_results[lags[0]]['r2'] else "递减"
        print(f"  R²从滞后{lags[0]}到滞后{lags[-1]}总体{r2_trend}")

# 文件输出总结
print("\\n=== 输出文件总结 ===")
print("生成的文件包括:")
print("1. training_results.csv - 训练结果统计")
print("2. predictions/predictions_lag_X.csv - 各滞后期的预测结果")
print("3. predictions/merged_predictions_TIMESTAMP.csv - 合并的预测结果")
print("4. models/ - 保存的模型文件")
print("5. training_results_visualization.png - 训练结果可视化")
print("6. prediction_comparison.png - 预测结果对比图")

print("\\n=== 任务完成 ===")
print("多滞后期XGBoost模型训练和预测任务已完成！")
